In [1]:
import sys
sys.path.insert(0, "..")

from optimal_long_short.model_params import KouParams
from optimal_long_short.kou_model import KouZTiltedDynamics
from optimal_long_short.root_finder import CharacteristicRootFinder


In [2]:
from optimal_long_short.market_params import MarketParams
from optimal_long_short.strategy import UnitExposureLongShortStrategy
from optimal_long_short.laplace_resolvent import ParticularSolution, HomogeneousSolution, GeneralSolution

# --- parameters ---
params = KouParams(
    mu1=0.05, sigma1=0.20, lam1=1.0, p1=0.5, eta1_pos=10.0, eta1_neg=8.0,
    mu2=0.03, sigma2=0.15, lam2=0.8, p2=0.5, eta2_pos=12.0, eta2_neg=9.0,
    rho=0.3,
)
market = MarketParams(b=0.8, S10=1.0, S20=1.0)

h0   = 0.5    # initial log-health
q    = 0.05   # Laplace parameter

for k in (1, 2):
    dyn      = KouZTiltedDynamics(params=params, k=k)
    strategy = UnitExposureLongShortStrategy(h0=h0, market=market)

    part  = ParticularSolution(dynamics=dyn, strategy=strategy)
    hom   = HomogeneousSolution(particular=part)
    sol   = GeneralSolution(homogeneous=hom)

    U_at_origin = sol.evaluate_at_origin(q)

    print(f"\n=== k={k}, h0={h0}, q={q} ===")
    print(f"  Forcing coefficients c_{{k,j}}:")
    for j, c in enumerate(part.coefficients()):
        print(f"    c_{{k,{j}}} = {c:.6f}")
    print(f"  Forcing vector b^(k):")
    for i, bi in enumerate(part.forcing_vector(q)):
        print(f"    b[{i}] = {bi:.6f}")
    print(f"  Barrier matrix M_bar:")
    M = hom.barrier_matrix(q)
    for row in M:
        print(f"    {[f'{v:.6f}' for v in row]}")
    print(f"  Homogeneous coefficients B:")
    B = hom.coefficients(q)
    for m, Bm in enumerate(B, 1):
        print(f"    B_{m} = {Bm:.6f}")
    print(f"  U_hat(q=0.05, z=0; h0={h0}) = {U_at_origin:.8f}")



=== k=1, h0=0.5, q=0.05 ===
  Forcing coefficients c_{k,j}:
    c_{k,0} = -0.800000
    c_{k,1} = 1.648721
  Forcing vector b^(k):
    b[0] = 40.774194+0.000000j
    b[1] = 34.465950+0.000000j
    b[2] = 36.043011+0.000000j
  Barrier matrix M_bar:
    ['1.000000+0.000000j', '1.000000+0.000000j', '1.000000+0.000000j']
    ['-1.425800+0.000000j', '-6.385135-0.000000j', '1.138364+0.000000j']
    ['-4.213128+0.000000j', '6.296184-0.000000j', '1.096969+0.000000j']
  Homogeneous coefficients B:
    B_1 = -2.392399-0.000000j
    B_2 = -0.772970+0.000000j
    B_3 = -37.608825-0.000000j
  U_hat(q=0.05, z=0; h0=0.5) = 54.46641986-0.00000000j

=== k=2, h0=0.5, q=0.05 ===
  Forcing coefficients c_{k,j}:
    c_{k,0} = 0.640000
    c_{k,1} = -2.637954
    c_{k,2} = 2.718282
  Forcing vector b^(k):
    b[0] = -45.646565+0.000000j
    b[1] = -37.072124+0.000000j
    b[2] = -38.560191+0.000000j
  Barrier matrix M_bar:
    ['1.000000+0.000000j', '1.000000+0.000000j', '1.000000+0.000000j']
    ['-1.5902